# Last.fm Scraper

**Run this notebook in 1 to 4 VSCode windows simultaneously.**

- All workers read/write one shared file: `data/lastfm_data.parquet`
- A file lock prevents two workers writing at the same time
- Duplicate rows are blocked on every write
- **Optional:** import an existing CSV into the parquet before scraping starts (Cell 3)

In [ ]:
import os, time, re
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from filelock import FileLock

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SET THIS — unique number for each VSCode window            ║
# ║  Window 1 → 0 | Window 2 → 1 | Window 3 → 2 | Window 4 → 3║
# ╚══════════════════════════════════════════════════════════════╝
WORKER_ID     = 0   # change to 1, 2, or 3 in each other window
TOTAL_WORKERS = 4   # set to however many windows you are running (1–4)
SAVE_EVERY    = 50  # write to parquet every N scraped rows

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
PARQUET_PATH = 'data/mb_album_artists.parquet'   # MusicBrainz source
OUT_PATH     = 'data/lastfm_data.parquet'         # single shared output
LOCK_PATH    = 'data/lastfm_data.parquet.lock'    # file lock (auto-created)

COLS = ['Artist', 'Album',
        'Artist_Listeners', 'Artist_Scrobbles',
        'Album_Listeners',  'Album_Scrobbles',
        'Similar_Artists',  'Artist_URL', 'Album_URL']

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}
lock = FileLock(LOCK_PATH, timeout=60)
print(f'Worker {WORKER_ID} ready   output: {OUT_PATH}')

## (Optional) Import existing CSV
If you already have scraped data in a CSV, run this cell once to import it into the shared parquet.  
**Skip this cell if you have no CSV** — it does nothing if `CSV_IMPORT_PATH` does not exist.

In [ ]:
CSV_IMPORT_PATH = 'data/lastfm_data.csv'   # <-- point to your CSV (or leave as-is to skip)

if not os.path.exists(CSV_IMPORT_PATH):
    print('No CSV found at', CSV_IMPORT_PATH, '— skipping import.')
else:
    df_csv = pd.read_csv(CSV_IMPORT_PATH)

    # Normalise column names — handle both "Similar Artists" and "Similar_Artists"
    df_csv.columns = [c.strip().replace(' ', '_') for c in df_csv.columns]
    df_csv = df_csv.rename(columns={
        'Similar_Artists': 'Similar_Artists',  # already correct
    })

    # Keep only the columns we need, fill any missing ones with N/A
    for col in COLS:
        if col not in df_csv.columns:
            df_csv[col] = 'N/A'
    df_csv = df_csv[COLS].copy()

    print(f'CSV rows loaded: {len(df_csv):,}')

    # Merge with existing parquet (if any) and deduplicate
    with lock:
        if os.path.exists(OUT_PATH):
            df_existing = pd.read_parquet(OUT_PATH)
            df_out = pd.concat([df_existing, df_csv], ignore_index=True)
        else:
            df_out = df_csv

        before = len(df_out)
        df_out = df_out.drop_duplicates(subset=['Artist', 'Album']).reset_index(drop=True)
        df_out.to_parquet(OUT_PATH, index=False)

    print(f'Imported {before - (before - len(df_out)):,} new rows  ({before - len(df_out):,} duplicates dropped)')
    print(f'Total rows in parquet now: {len(df_out):,}')

In [ ]:
# ── Load MusicBrainz source & compute this worker's slice ────────────────────
df_all = pd.read_parquet(PARQUET_PATH)
artist_col = 'artist_name' if 'artist_name' in df_all.columns else 'name'
album_col  = 'album_name'  if 'album_name'  in df_all.columns else 'album'

df_unique = (
    df_all[[artist_col, album_col]]
    .drop_duplicates()
    .reset_index(drop=True)
)

total    = len(df_unique)
chunk    = (total + TOTAL_WORKERS - 1) // TOTAL_WORKERS
start    = WORKER_ID * chunk
end      = min(start + chunk, total)
df_slice = df_unique.iloc[start:end].reset_index(drop=True)

print(f'Total unique albums : {total:,}')
print(f'Worker {WORKER_ID} slice   : rows {start:,} to {end:,}  ({len(df_slice):,} albums)')

In [ ]:
# ── Build done-set from the shared parquet ───────────────────────────────────
def load_done_set():
    """Returns a set of (artist.lower(), album.lower()) already in the parquet."""
    if not os.path.exists(OUT_PATH):
        return set()
    df_done = pd.read_parquet(OUT_PATH, columns=['Artist', 'Album'])
    return set(zip(df_done['Artist'].str.lower(), df_done['Album'].str.lower()))

done_set = load_done_set()

df_todo = df_slice[
    ~df_slice.apply(
        lambda r: (str(r[artist_col]).lower(), str(r[album_col]).lower()) in done_set,
        axis=1
    )
].reset_index(drop=True)

print(f'Already scraped in this slice : {len(df_slice) - len(df_todo):,}')
print(f'Remaining to scrape           : {len(df_todo):,}')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def scrape_artist(url):
    listeners, scrobbles, similar = 'N/A', 'N/A', 'None Found'
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, 'html.parser')
                container = soup.find('div', class_='header-new-info-desktop')
                if container:
                    for item in container.find_all('li', class_='header-metadata-tnew-item'):
                        title = item.find('h4', class_='header-metadata-tnew-title')
                        abbr  = item.find('abbr', class_='js-abbreviated-counter')
                        if title and abbr:
                            if 'Listeners'  in title.text: listeners = abbr.get('title')
                            elif 'Scrobbles' in title.text: scrobbles = abbr.get('title')
                sims = [a.text.strip()
                        for h in soup.find_all('h3', class_='catalogue-overview-similar-artists-item-name')
                        for a in [h.find('a')] if a]
                if sims: similar = ', '.join(sims)
                if listeners != 'N/A': break
            time.sleep(2)
        except Exception as e:
            print(f'   artist error ({attempt+1}/3): {e}')
            time.sleep(2)
    return listeners, scrobbles, similar

def scrape_album(url):
    listeners, scrobbles = 'N/A', 'N/A'
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                soup  = BeautifulSoup(r.text, 'html.parser')
                abbrs = soup.find_all('abbr', class_='js-abbreviated-counter')
                if len(abbrs) >= 2:
                    listeners = abbrs[0].get('title', 'N/A')
                    scrobbles = abbrs[1].get('title', 'N/A')
                elif len(abbrs) == 1:
                    listeners = abbrs[0].get('title', 'N/A')
                if listeners != 'N/A' and scrobbles != 'N/A': break
            time.sleep(2)
        except Exception as e:
            print(f'   album error ({attempt+1}/3): {e}')
            time.sleep(2)
    return listeners, scrobbles

def flush_buffer(buffer):
    """Append buffer to the shared parquet safely using a file lock."""
    df_new = pd.DataFrame(buffer, columns=COLS)
    with lock:
        if os.path.exists(OUT_PATH):
            df_out = pd.concat([pd.read_parquet(OUT_PATH), df_new], ignore_index=True)
        else:
            df_out = df_new
        df_out = df_out.drop_duplicates(subset=['Artist', 'Album']).reset_index(drop=True)
        df_out.to_parquet(OUT_PATH, index=False)
    return len(df_out)

In [ ]:
# ── Main scraping loop ───────────────────────────────────────────────────────
print(f'Worker {WORKER_ID} starting — {len(df_todo):,} albums to scrape')
print('-' * 60)

buffer           = []
scraped_this_run = 0
skipped_this_run = 0

try:
    for i, row in df_todo.iterrows():
        artist = str(row[artist_col]).strip()
        album  = str(row[album_col]).strip()

        if not artist or artist.lower() in ('none', 'nan'):
            continue

        # ── Refresh done_set every 500 rows (picks up other workers' progress) ─
        if scraped_this_run > 0 and scraped_this_run % 500 == 0:
            done_set = load_done_set()

        # ── CHECK before scraping ────────────────────────────────────────────
        if (artist.lower(), album.lower()) in done_set:
            skipped_this_run += 1
            print(f'[W{WORKER_ID}] [{i+1}/{len(df_todo)}] SKIP: {artist} — {album}')
            continue

        # ── Scrape ───────────────────────────────────────────────────────────
        a_slug     = artist.replace(' ', '+')
        al_slug    = album.replace(' ', '+')
        artist_url = f'https://www.last.fm/music/{a_slug}'
        album_url  = f'https://www.last.fm/music/{a_slug}/{al_slug}'

        print(f'[W{WORKER_ID}] [{i+1}/{len(df_todo)}] {artist} — {album}')

        a_listeners, a_scrobbles, similar = scrape_artist(artist_url)
        time.sleep(1)
        al_listeners, al_scrobbles = scrape_album(album_url)

        buffer.append([artist, album,
                        a_listeners, a_scrobbles,
                        al_listeners, al_scrobbles,
                        similar, artist_url, album_url])

        done_set.add((artist.lower(), album.lower()))  # local update
        scraped_this_run += 1

        # ── Flush buffer to parquet every SAVE_EVERY rows ───────────────────
        if len(buffer) >= SAVE_EVERY:
            total_saved = flush_buffer(buffer)
            buffer = []
            print(f'   [W{WORKER_ID}] Saved — {total_saved:,} total rows in parquet')

        time.sleep(1.5)

    print(f'\nWorker {WORKER_ID} DONE!')
    print(f'  Scraped : {scraped_this_run:,}')
    print(f'  Skipped : {skipped_this_run:,}')

except KeyboardInterrupt:
    print(f'\nWorker {WORKER_ID} stopped — {scraped_this_run:,} rows scraped this run.')

finally:
    if buffer:
        total_saved = flush_buffer(buffer)
        buffer = []
        print(f'Final flush: {total_saved:,} total rows in parquet')

---
## Status — run from any window at any time

In [ ]:
if os.path.exists(OUT_PATH):
    df_status = pd.read_parquet(OUT_PATH)
    print(f'Total rows in parquet : {len(df_status):,}')
    print(f'Unique artists        : {df_status["Artist"].nunique():,}')
    print(f'File size             : {os.path.getsize(OUT_PATH)/1024:.1f} KB')
    display(df_status.tail(5))
else:
    print('Parquet not created yet — run the scraper first.')